# Probabilidad

Lo que hicimos hasta acá miraba hacia atrás: las esperas de la clínica
que **ya pasaron**, los conteos de defectos del turno **ya
inspeccionado**, las respuestas que la encuesta **ya recogió**.
Apenas el contexto se mueve un poco — un paciente nuevo cruza la
puerta, arranca un turno todavía sin medir, alguien levanta la mano
por primera vez —, los promedios y desvíos del capítulo anterior
dejan de alcanzar.

¿Qué tan probable es que el próximo paciente espere más de cinco
minutos?, si ya lleva tres minutos sentado, ¿cómo se actualizan esas
chances?, cuando alguien dice «sí» en la encuesta, ¿qué tanto pesa esa
respuesta sobre lo que vamos a afirmar del barrio entero? Las tres
preguntas comparten la misma maquinaria: el **álgebra de eventos**
(uniones, intersecciones, complementos), la **probabilidad
condicional** que mide cuánto se mueve una creencia frente a
información nueva, y el **teorema de Bayes** que invierte ese
movimiento para preguntar al revés.

## Importaciones

In [ ]:
from core import Settings
from exercises import NumericAnswerInput, verify_numeric_answer
from probability import (
    BayesInput,
    ConditionalInput,
    JointEventInput,
    SetOperationInput,
    evaluate_bayes,
    evaluate_conditional_probability,
    evaluate_set_operations,
    joint_event_probabilities,
)
from probability.total_probability import TotalProbabilityBranch
from symbolic import bayes_theorem, total_probability_theorem
from widgets import BayesExplorerInput, build_bayes_explorer
from visualization import (
    PartitionDiagramInput,
    ProbabilityTreeInput,
    VennTwoSetsInput,
    chart_partition_diagram,
    chart_probability_tree,
    chart_venn_two_sets,
)

In [ ]:
settings = Settings()

(sec-prob-experiment)=
## Un experimento concreto

Antes de hablar de tiempos continuos, fijamos un experimento minimalista: tirar
un dado equilibrado. Definimos dos eventos:

- $A$ = «sale par» = $\{2, 4, 6\}$.
- $B$ = «sale al menos 4» = $\{4, 5, 6\}$.

Sobre este universo de 6 resultados podemos visualizar las operaciones de
conjuntos sin ambigüedad. Después transferimos la idea a eventos sobre
tiempos o resultados de la encuesta.


In [ ]:
universe = frozenset({"1", "2", "3", "4", "5", "6"})
event_even = frozenset({"2", "4", "6"})
event_at_least_four = frozenset({"4", "5", "6"})

set_input = SetOperationInput(
    universe=universe,
    event_a=event_even,
    event_b=event_at_least_four,
)
set_result = evaluate_set_operations(set_input)
set_result

(sec-prob-additive)=
## La regla aditiva

**Paso 1 — Cardinalidades.** Imaginá dos círculos solapados (Venn): $A \cap B$
vive en la zona compartida. Si sumamos cardinalidades a lo bestia, la zona
compartida se cuenta dos veces:

$$ |A \cup B| = |A| + |B| - |A \cap B| $$ (eq-union-card)

**Paso 2 — Probabilidades.** Dividiendo cada cardinalidad por $|\Omega|$
obtenemos la **regla aditiva** sobre probabilidades. Esta es la primera
fórmula propiamente probabilística del libro:

$$ P(A \cup B) = P(A) + P(B) - P(A \cap B) $$ (eq-union-prob)

Volviendo al dado: $P(A) = P(B) = 3/6$, $P(A \cap B) = 2/6$, así que la
ecuación [](#eq-union-prob) da $P(A \cup B) = 3/6 + 3/6 - 2/6 = 4/6$.


In [ ]:
joint_input = JointEventInput(
    probability_a=3 / 6,
    probability_b=3 / 6,
    probability_intersection=2 / 6,
)
joint = joint_event_probabilities(joint_input)
joint

Para ver por qué la cardinalidad del solapamiento se cuenta dos veces, conviene
tener el diagrama a mano. Cada círculo es un evento, la región compartida es la
intersección $A \cap B$, y la unión es todo lo coloreado.


In [ ]:
venn_input = VennTwoSetsInput(
    set_a_label="A: sale par",
    set_b_label="B: sale al menos 4",
    title="Eventos sobre el dado",
    settings=settings,
)
chart_venn_two_sets(venn_input)

(sec-prob-conditional)=
## Probabilidad condicional

Probabilidad condicional es **restringir el universo**: dentro del subconjunto
$B$, ¿qué fracción cae también en $A$? **Paso 1:** retomamos $P(A \cap B)$ que
ya definimos junto a [](#eq-union-prob). **Paso 2:** definimos:

$$ P(A \mid B) = \frac{P(A \cap B)}{P(B)},\quad P(B) > 0 $$ (eq-cond-prob)

Con los datos del dado: $P(A \mid B) = (2/6)/(3/6) = 2/3$. Saber que el
resultado es al menos 4 sube la chance de «par» de $1/2$ a $2/3$.


In [ ]:
conditional_input = ConditionalInput(
    probability_intersection=2 / 6,
    probability_conditioning_event=3 / 6,
)
conditional = evaluate_conditional_probability(conditional_input)
conditional

(sec-prob-events-as-waits)=
## Las esperas vistas como eventos

Volvemos a la sala de espera y miramos la fórmula [](#eq-cond-prob) en ese contexto. Tenemos
$T$ = «espera más de 5 minutos» y $S$ = «espera más de 3 minutos». Por
construcción $T \subset S$, así que $P(T \cap S) = P(T)$. La fórmula [](#eq-cond-prob)
queda:

$$ \begin{aligned}
P(T \mid S) &= \frac{P(T \cap S)}{P(S)} \\[4pt]
            &= \frac{P(T)}{P(S)}
\end{aligned} $$

Si en una semana de datos pasados $P(T) = 0{,}21$ y $P(S) = 0{,}45$,
entonces $P(T \mid S) = 0{,}21 / 0{,}45 \approx 0{,}47$. Ya esperar 3 minutos
casi duplica la chance original de cruzar los 5.


In [ ]:
clinic_conditional_input = ConditionalInput(
    probability_intersection=0.21,
    probability_conditioning_event=0.45,
)
clinic_conditional = evaluate_conditional_probability(clinic_conditional_input)
clinic_conditional

(sec-prob-bayes-symbolic)=
## Teorema de Bayes (forma simbólica)

**Paso 1.** Escribimos la condicional en los dos sentidos usando [](#eq-cond-prob):

$$ P(A \cap B) = P(A \mid B)\,P(B) = P(B \mid A)\,P(A) $$

**Paso 2.** Igualamos los dos miembros de la derecha y despejamos
$P(A \mid B)$:

$$ P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)} $$ (eq-bayes)

Dejamos también la versión simbólica visible para que la forma sea
inequívoca:


In [ ]:
bayes_theorem().formula

(sec-prob-bayes-data)=
## Bayes con datos: prueba diagnóstica

**Concreto.** En una población, $P(D) = 0{,}01$ es la prevalencia. El test
tiene sensibilidad $P(+ \mid D) = 0{,}99$ y especificidad
$P(- \mid \bar{D}) = 0{,}95$ (ergo $P(+ \mid \bar{D}) = 0{,}05$). Buscamos
$P(D \mid +)$.

**Pictórico.** Pensá en 10.000 personas. 100 enfermos: 99 dan positivo.
9.900 sanos: 495 dan positivo. Total de positivos: $99 + 495 = 594$. La
fracción de verdaderos enfermos dentro de los positivos es
$99 / 594 \approx 0{,}167$.

**Abstracto.** Aplicamos [](#eq-bayes) con la **probabilidad total** en el
denominador. Para no sufrir scroll horizontal, expresamos la cuenta en
varias líneas:

$$ \begin{aligned}
P(D \mid +) &= \frac{P(+ \mid D)\,P(D)}{P(+)} \\[4pt]
P(+)        &= P(+ \mid D)\,P(D) + P(+ \mid \bar{D})\,P(\bar{D}) \\[4pt]
            &= 0{,}99 \cdot 0{,}01 + 0{,}05 \cdot 0{,}99 \\[4pt]
            &= 0{,}0099 + 0{,}0495 \\[4pt]
            &= 0{,}0594 \\[4pt]
P(D \mid +) &= \frac{0{,}0099}{0{,}0594} \approx 0{,}167
\end{aligned} $$


In [ ]:
diagnostic_branches = (
    TotalProbabilityBranch(label="Enfermo", prior=0.01, likelihood=0.99),
    TotalProbabilityBranch(label="Sano", prior=0.99, likelihood=0.05),
)
diagnostic_input = BayesInput(branches=diagnostic_branches)
diagnostic_posteriors = evaluate_bayes(diagnostic_input)
diagnostic_posteriors[0]

El mismo árbol que dibujarías a mano resume las cuatro ramas posibles. Cada
rama parte de la prevalencia y se bifurca según el resultado del test; la hoja
lleva la **probabilidad conjunta**, que es justo lo que [](#eq-total-prob) pide sumar para
armar $P(+)$.


In [ ]:
tree_input = ProbabilityTreeInput(
    root_label="Población",
    branch_labels=("Enfermo", "Sano"),
    branch_probabilities=(0.01, 0.99),
    leaf_labels=("Test +", "Test −"),
    conditional_probabilities=((0.99, 0.01), (0.05, 0.95)),
    title="Árbol de la prueba diagnóstica",
    settings=settings,
)
chart_probability_tree(tree_input)

Aunque el test es excelente, un positivo apenas mueve la creencia del 1% al
16,7%. La culpa la tiene la **tasa base**: como casi nadie está enfermo, los
falsos positivos pesan mucho. Movés los sliders del widget y vas a ver cómo
el posterior reacciona.


In [ ]:
explorer_input = BayesExplorerInput(settings=settings)
build_bayes_explorer(explorer_input)

(sec-prob-total)=
## Probabilidad total — la versión general

Si $\{A_1, \dots, A_k\}$ es una **partición** del universo (eventos
incompatibles que cubren todo), la probabilidad total da el denominador
limpio que usamos en el ejemplo anterior:

$$ P(B) = \sum_{i=1}^{k} P(B \mid A_i)\,P(A_i) $$ (eq-total-prob)


In [ ]:
total_probability_theorem(partition_size=3).formula

Una **partición** se ve como una franja $\Omega$ cortada en pedazos disjuntos
$A_1, \dots, A_k$. La barra inferior anaranjada muestra cómo el evento $B$ se
reparte dentro de cada pedazo: la suma de esos cachitos, ponderada por
$P(A_i)$, es exactamente la fórmula [](#eq-total-prob).


In [ ]:
partition_input = PartitionDiagramInput(
    partition_labels=("A_1", "A_2", "A_3"),
    partition_weights=(0.45, 0.35, 0.20),
    overlay_label="B se reparte sobre la partición",
    overlay_fractions=(0.30, 0.55, 0.10),
    title="Partición de Ω y evento B",
    settings=settings,
)
chart_partition_diagram(partition_input)

(sec-prob-inspection)=
## Una pieza pasa la inspección

La línea de producción inspecciona cada pieza con un test que detecta
defectos con probabilidad $0{,}9$ si la pieza está mal y produce un falso
positivo con probabilidad $0{,}05$ si está bien. La verdadera tasa de piezas
defectuosas es $0{,}03$.

Aplicamos [](#eq-bayes) y [](#eq-total-prob) para responder: **dada una alarma, ¿cuál es la
probabilidad de que la pieza esté realmente defectuosa?**


In [ ]:
factory_branches = (
    TotalProbabilityBranch(label="Defectuosa", prior=0.03, likelihood=0.90),
    TotalProbabilityBranch(label="Sana", prior=0.97, likelihood=0.05),
)
factory_input = BayesInput(branches=factory_branches)
factory_posteriors = evaluate_bayes(factory_input)
factory_posteriors[0]

## Ejercicio 1 — Regla aditiva

Sean $P(A) = 0{,}6$, $P(B) = 0{,}5$ y $P(A \cap B) = 0{,}2$. Calculá
$P(A \cup B)$ usando la fórmula [](#eq-union-prob):

$$ P(A \cup B) = 0{,}6 + 0{,}5 - 0{,}2 = 0{,}9 $$


In [ ]:
exercise_joint_input = JointEventInput(
    probability_a=0.6,
    probability_b=0.5,
    probability_intersection=0.2,
)
expected_union = joint_event_probabilities(exercise_joint_input).union

student_answer_union = 0.9
verify_input = NumericAnswerInput(
    student_answer=student_answer_union,
    expected_answer=expected_union,
)
verify_numeric_answer(verify_input)

## Ejercicio 2 — Bayes a mano

Una caja $C_1$ tiene 3 bolas rojas y 7 blancas. La caja $C_2$ tiene 6 rojas
y 4 blancas. Se elige una caja al azar y se saca una bola: resulta roja.
Calculá $P(C_1 \mid R)$ aplicando la fórmula [](#eq-bayes) con denominador [](#eq-total-prob):

$$ \begin{aligned}
P(C_1 \mid R) &= \frac{P(R \mid C_1)\,P(C_1)}{P(R)} \\[4pt]
P(R)          &= 0{,}3 \cdot 0{,}5 + 0{,}6 \cdot 0{,}5 \\[4pt]
              &= 0{,}45 \\[4pt]
P(C_1 \mid R) &= \frac{0{,}3 \cdot 0{,}5}{0{,}45} \\[4pt]
              &= \frac{1}{3}
\end{aligned} $$


In [ ]:
box_branches = (
    TotalProbabilityBranch(label="C1", prior=0.5, likelihood=0.3),
    TotalProbabilityBranch(label="C2", prior=0.5, likelihood=0.6),
)
box_input = BayesInput(branches=box_branches)
expected_posterior = evaluate_bayes(box_input)[0].posterior

student_answer_posterior = 1 / 3
verify_input = NumericAnswerInput(
    student_answer=student_answer_posterior,
    expected_answer=expected_posterior,
)
verify_numeric_answer(verify_input)

Las probabilidades que armamos hasta acá viven sobre conjuntos discretos: un
dado, una caja, un test que da positivo o negativo. Sin embargo muy pronto
vamos a tener que asignar probabilidad a cosas que ni siquiera se pueden
enumerar — el tiempo exacto de espera de un paciente, la altura precisa de una
persona, el conteo de defectos en un turno cualquiera. ¿Cómo se le asigna
probabilidad a un intervalo de minutos sin listar todos los resultados
posibles? ¿Qué objeto matemático es capaz de modelar «cantidad de piezas
defectuosas en un turno» de modo que ya tenga, por construcción, una media y
una varianza? Ese objeto se llama **variable aleatoria**, y su PMF y su
densidad son las herramientas que aparecen a continuación.
